# Real-model evaluation in Google Colab

This notebook runs the existing Phase 7/8 Apertus Eval Prep platform against one selected open-weight Transformers model on an interactive GPU runtime. It is an evaluation experiment, not foundation-model training or fine-tuning.

This produces **experimental real-model evidence**, not a production benchmark. Colab hardware is ephemeral and can vary by session. Results apply only to the recorded model/tokenizer revisions, prompt, dataset, decoding, and runtime. Do not commit tokens, secrets, private datasets, downloaded model caches, or large generated runs. Release-gate results are engineering policy aids and are not production approval.

In [ ]:
import importlib.metadata as importlib_metadata
import json
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_PATH = "/content/apertus-eval-prep"
repo = Path(REPOSITORY_PATH).expanduser()
if not (repo / "pyproject.toml").exists():
    repository_url = os.environ.get("APERTUS_REPO_URL", "https://github.com/Shivani767/apertus-eval-prep.git")
    if not repository_url:
        raise RuntimeError("Open/clone the repository first or set APERTUS_REPO_URL privately.")
    subprocess.run(["git", "clone", "--depth", "1", repository_url, str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[dev]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[real-model]"], check=True)
import apertus_eval_prep
print("apertus_eval_prep", apertus_eval_prep.__version__)
for package in ("torch", "transformers", "accelerate", "pyyaml"):
    try:
        print(package, importlib_metadata.version(package))
    except importlib_metadata.PackageNotFoundError:
        print(package, "UNAVAILABLE")
subprocess.run([sys.executable, "-m", "apertus_eval_prep", "platform-run", "--config", "configs/platform_smoke.yaml", "--out", "runs/phase8_real_colab/mock_smoke"], check=True)
print("Offline mock smoke completed. Expected evidence mode: MOCK.")

In [ ]:
import json
# pyright: reportUndefinedVariable=false
from apertus_eval_prep.utils.runtime_profile import profile_runtime

# Prefer the config-cell values, but never crash if that cell was not run.
_cfg = {key: globals().get(key) for key in ('DEVICE', 'DTYPE', 'QUANTIZATION')}
DEVICE = _cfg['DEVICE'] or 'cuda'
DTYPE = _cfg['DTYPE'] or 'float16'
QUANTIZATION = _cfg['QUANTIZATION'] or 'none'
print('config inherited:', {k: v for k, v in _cfg.items() if v is not None} or 'none (defaults used)')

runtime_profile = profile_runtime(device=DEVICE, precision=DTYPE, quantization=QUANTIZATION)

gpu_name = runtime_profile.get('gpu_name')
gpu_count = runtime_profile.get('gpu_count') or 0
torch_version = str(runtime_profile.get('torch_version') or '')
gpu_memory = runtime_profile.get('gpu_memory')
gpu_unit = runtime_profile.get('gpu_memory_unit') or ''
status = 'GPU READY' if (gpu_name and gpu_count) else 'NO GPU'

print('[%s] device=%s dtype=%s quantization=%s' % (status, DEVICE, DTYPE, QUANTIZATION))
print('  torch      :', torch_version or 'not installed')
print('  cuda       :', runtime_profile.get('cuda_version'))
print('  gpu        :', '%s x%s' % (gpu_name or 'none', gpu_count))
print('  gpu memory :', ('%s %s' % (gpu_memory, gpu_unit)).strip() or 'unknown')
print('  cpu count  :', runtime_profile.get('cpu_count'))
print('  environment:', runtime_profile.get('runtime_environment'))
print('  measured   :', runtime_profile.get('hardware_measurement_status'))

if not gpu_name or not gpu_count:
    print()
    print('!! No GPU detected. The real-model cells will refuse to run.')
    print('   Fix: Runtime -> Change runtime type -> GPU (T4/L4), then')
    print('        Runtime -> Restart session, and re-run this cell.')
    if '+cpu' in torch_version:
        print('   NOTE: a CPU-only torch build is installed. After enabling a GPU,')
        print('         do NOT reinstall torch or pip will fetch another +cpu wheel.')
        print('         Use instead: pip install transformers accelerate')
elif '+cpu' in torch_version:
    print()
    print('!! GPU is present but torch is a CPU-only build (+cpu).')
    print('   Install the CUDA build, e.g.:')
    print('     pip install torch --index-url https://download.pytorch.org/whl/cu121')
    print('   then restart the session and re-run this cell.')

print()
print('Full profile:')
print(json.dumps(runtime_profile, indent=2, default=str))
print()
print('Colab hardware is ephemeral and may differ between sessions.')
print('Do not treat this profile as production serving hardware.')

## 1. Select the model and decoding settings

Edit only the variables in the next cell. Use a smaller model if GPU memory is limited. Begin with float16/no quantization. Use int8/int4 only after the baseline succeeds and optional quantization dependencies are installed. Pin a model revision if available. Do not enable `trust_remote_code` unless required and reviewed. No access token belongs in this notebook.

In [ ]:
MODEL_ID = "YOUR_MODEL_ID"
MODEL_REVISION = "OPTIONAL_PINNED_REVISION"
TOKENIZER_ID = None
DEVICE = "cuda"
DTYPE = "float16"
QUANTIZATION = "none"
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.0
TOP_P = 1.0
SEED = 11

In [ ]:
# pyright: reportUndefinedVariable=false
import json
import subprocess
import sys
from pathlib import Path
from apertus_eval_prep.utils.runtime_profile import profile_runtime
from apertus_eval_prep.utils.serialization import read_yaml, write_yaml

OUTPUT_ROOT = Path("runs/phase8_real_colab")
CONFIG_ROOT = OUTPUT_ROOT / "configs"
CONFIG_ROOT.mkdir(parents=True, exist_ok=True)
STUDY_DIR = Path("configs/studies/phase8_sarvam_application_study")
PRIVATE_CONFIG = STUDY_DIR / "local_user_config.yaml"
TEMPLATE = STUDY_DIR / "local_user_config.example.yaml"
REVISION = None if str(MODEL_REVISION).startswith("OPTIONAL_") else MODEL_REVISION
TOKENIZER = TOKENIZER_ID or MODEL_ID
if str(MODEL_ID).startswith("YOUR_") or not str(MODEL_ID).strip():
    raise RuntimeError("Edit MODEL_ID before running the real-model cells.")
if DEVICE != "cuda":
    raise RuntimeError("This workflow is intentionally CUDA-only; it will not silently fall back to CPU.")

def require_cuda():
    profile = profile_runtime(device=DEVICE, precision=DTYPE, quantization=QUANTIZATION)
    print(json.dumps(profile, indent=2, default=str))
    if not profile.get("gpu_name") or not profile.get("gpu_count"):
        raise RuntimeError("CUDA GPU is unavailable. Enable a Colab GPU runtime before continuing.")

def _base(config):
    return config["experiment"]["base"] if "experiment" in config else config

def _set_identity(config, output_dir):
    base = _base(config)
    adapter = base.setdefault("adapter", {})
    adapter.update(kind="local_transformers", model_id=MODEL_ID, revision=REVISION)
    params = adapter.setdefault("params", {})
    params.update(tokenizer_id=TOKENIZER, tokenizer_revision=REVISION, trust_remote_code=False, device=DEVICE, dtype=DTYPE, quantization=QUANTIZATION, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE, top_p=TOP_P, do_sample=TEMPERATURE > 0.0, timeout_s=600)
    base.setdefault("runtime", {}).update(device=DEVICE, precision=DTYPE, quantization=QUANTIZATION, timeout_s=600)
    base.setdefault("decoding", {}).update(seed=SEED, temperature=TEMPERATURE, top_p=TOP_P, max_new_tokens=MAX_NEW_TOKENS)
    base.setdefault("cost", {}).update(input_per_million=None, output_per_million=None, currency="USD", source="manual_config", effective_date=None, estimate_label="DERIVED_ESTIMATE")
    base.setdefault("evidence", {}).update(mode="LOCAL_REAL_MODEL", runtime_environment="google_colab", real_model_execution=True, external_provider_execution=False, hardware_measured=True, human_reviewed=False, pricing_source="manual_config")
    base.setdefault("reporting", {})["output_dir"] = str(output_dir)
    if isinstance(config.get("model"), dict):
        config["model"].update(id=MODEL_ID, revision=REVISION, tokenizer_id=TOKENIZER)
    if "experiment" in config:
        experiment = config["experiment"]
        experiment["baseline"].update(seed=SEED, temperature=TEMPERATURE, top_p=TOP_P, prompt_template="base", model_revision=REVISION, backend="local_transformers", precision=DTYPE, quantization=QUANTIZATION)
        experiment.setdefault("factors", {}).update(model_revision=[REVISION], backend=["local_transformers"], precision=[DTYPE], quantization=[QUANTIZATION], seed=[SEED], temperature=[TEMPERATURE], top_p=[TOP_P])
        base["prompt"] = {"prompt_id": "sarvam-core", "version": "v1", "template": "base"}
    return config

def write_private_config():
    write_yaml(PRIVATE_CONFIG, _set_identity(read_yaml(TEMPLATE), OUTPUT_ROOT / "local_baseline"))
    return PRIVATE_CONFIG

def write_experiment_config(template_name, destination, output_dir, *, seeds=None, prompt_templates=None):
    config = _set_identity(read_yaml(STUDY_DIR / template_name), output_dir)
    if "experiment" in config:
        if seeds is not None:
            config["experiment"].setdefault("factors", {})["seed"] = list(seeds)
            config["experiment"]["baseline"]["seed"] = seeds[0]
        if prompt_templates is not None:
            config["experiment"].setdefault("factors", {})["prompt_template"] = list(prompt_templates)
            config["experiment"]["baseline"]["prompt_template"] = prompt_templates[0]
    write_yaml(destination, config)
    return destination

def run_platform(*args):
    try:
        subprocess.run([sys.executable, "-m", "apertus_eval_prep", *map(str, args)], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError("Platform command failed. For GPU OOM, choose a smaller model or lower MAX_NEW_TOKENS/batch size; use int8/int4 only after the baseline succeeds and dependencies are installed. No condition was silently changed.") from exc

def run_dirs(root):
    return sorted({p.parent for p in Path(root).rglob("manifest.json") if p.is_file()})

def validate_real_run(directory):
    directory = Path(directory)
    manifest = json.loads((directory / "manifest.json").read_text(encoding="utf-8"))
    evidence = manifest.get("evidence") or {}
    if evidence.get("mode") != "LOCAL_REAL_MODEL" or evidence.get("real_model_execution") is not True:
        raise RuntimeError(f"Unexpected evidence metadata in {directory}: {evidence}")
    if not (manifest.get("model") or {}).get("model_id"):
        raise RuntimeError(f"Model identity missing from {directory}")
    adapter = ((manifest.get("extra") or {}).get("adapter") or {})
    profile = adapter.get("runtime_profile") or manifest.get("runtime_profile") or (manifest.get("environment") or {}).get("runtime_profile")
    if not profile:
        raise RuntimeError(f"Runtime profile missing from {directory}")
    for name in ("report.md", "report.html", "metrics.json", "config.resolved.yaml", "confidence_intervals.json", "failures.jsonl", "failure_fingerprint.json", "gate_report.json"):
        if not (directory / name).exists():
            raise RuntimeError(f"Required artifact missing: {directory / name}")
    return directory

write_private_config()

## 2. Core real-model workflow

The next cell runs the selected local model smoke, the minimal six-condition variance matrix (three seeds by two prompt templates), the English RAG/agent suite, and sanitized safety. The offline mock smoke was run in setup. This workflow will stop rather than use CPU when CUDA is unavailable.

In [ ]:
# pyright: reportUndefinedVariable=false
require_cuda()
run_platform("platform-run", "--config", PRIVATE_CONFIG, "--out", OUTPUT_ROOT / "local_baseline")
baseline_runs = run_dirs(OUTPUT_ROOT / "local_baseline")
if not baseline_runs:
    raise RuntimeError("No real baseline artifact was produced.")
validate_real_run(baseline_runs[-1])
variance_config = write_experiment_config("local_variance.yaml", CONFIG_ROOT / "local_variance.yaml", OUTPUT_ROOT / "local_variance", seeds=[11, 22, 33], prompt_templates=["base", "strict"])
run_platform("platform-matrix", "--config", variance_config, "--out", OUTPUT_ROOT / "local_variance")
rag_config = write_experiment_config("local_rag_agent.yaml", CONFIG_ROOT / "local_rag_agent.yaml", OUTPUT_ROOT / "local_rag_agent")
run_platform("platform-matrix", "--config", rag_config, "--out", OUTPUT_ROOT / "local_rag_agent")
safety_config = write_experiment_config("local_safety.yaml", CONFIG_ROOT / "local_safety.yaml", OUTPUT_ROOT / "local_safety")
run_platform("platform-safety", "--config", safety_config, "--out", OUTPUT_ROOT / "local_safety")
CORE_RUNS = []
for directory in run_dirs(OUTPUT_ROOT / "local_variance"):
    manifest = json.loads((directory / "manifest.json").read_text(encoding="utf-8"))
    if (manifest.get("conditions") or {}).get("prompt_template") == "base":
        validate_real_run(directory)
        CORE_RUNS.append(directory)
CORE_RUNS = sorted(set(CORE_RUNS))
if len(CORE_RUNS) < 2:
    raise RuntimeError("At least two compatible base-template real core runs are required for selection.")
print("compatible real core runs:", [str(p) for p in CORE_RUNS])

## 3. Optional India-context diagnostic

Run only after core English results are complete. This is an exploratory diagnostic, not a multilingual benchmark, and it must not replace the English-first primary results.

In [ ]:
# pyright: reportUndefinedVariable=false
RUN_INDIA_CONTEXT_DIAGNOSTIC = False
if RUN_INDIA_CONTEXT_DIAGNOSTIC:
    require_cuda()
    india_config = write_experiment_config("india_context_diagnostic.yaml", CONFIG_ROOT / "india_context_diagnostic.yaml", OUTPUT_ROOT / "india_context_diagnostic")
    run_platform("platform-run", "--config", india_config, "--out", OUTPUT_ROOT / "india_context_diagnostic")
else:
    print("Skipped optional India-context diagnostic; core English results are the primary study.")

## 4. Ingest, select, report, review, and analyze

Only real LOCAL_REAL_MODEL core runs are passed to the primary deployment comparison. Mock smoke artifacts remain separate under mock_smoke. Do not use --allow-incompatible.

In [ ]:
# pyright: reportUndefinedVariable=false
import json
from pathlib import Path
from apertus_eval_prep.utils.serialization import read_yaml, write_yaml

points_path = OUTPUT_ROOT / "comparison_points.json"
selection_path = OUTPUT_ROOT / "selection.json"
run_platform("platform-ingest-runs", "--runs", *map(str, CORE_RUNS), "--out", points_path)
run_platform("platform-select", "--points", points_path, "--constraints", STUDY_DIR / "selection_constraints.json", "--out", selection_path)
report_run = CORE_RUNS[0]
run_platform("platform-report", "--run", report_run, "--format", "both", "--out", report_run)
run_platform("platform-fingerprint", "--run", report_run)
review_package = OUTPUT_ROOT / "review_package.jsonl"
run_platform("platform-export-review", "--run", report_run, "--dimensions", "correctness", "groundedness", "safe_behavior", "instruction_following", "tool_use_correctness", "--sample-size", "30", "--sampling-strategy", "stratified", "--seed", "7", "--study-id", "sarvam_application_real_eval_v1", "--out", review_package)
print("Review package written. Templates and empty packages are not human evidence.")
completed_annotations = Path("YOUR_COMPLETED_ANNOTATIONS.jsonl")
review_results = OUTPUT_ROOT / "review_results.json"
if completed_annotations.exists():
    run_platform("platform-ingest-review", "--input", completed_annotations, "--study-id", "sarvam_application_real_eval_v1", "--out", review_results)
study_config = read_yaml(STUDY_DIR / "study_metadata.yaml")
study_config.setdefault("study", {})["matrix"] = {"models": [MODEL_ID], "prompt_templates": ["base", "strict"], "seeds": [11, 22, 33], "dtype": [DTYPE], "quantization": [QUANTIZATION], "decoding": {"temperature": [0.0], "top_p": [1.0]}}
study_config["base"] = _set_identity({"base": study_config.get("base", {})}, OUTPUT_ROOT / "study")["base"]
private_study = CONFIG_ROOT / "study_metadata.yaml"
write_yaml(private_study, study_config)
study_args = ["platform-study-analyze", "--study-config", private_study, "--runs", *map(str, CORE_RUNS), "--out", OUTPUT_ROOT / "study"]
if review_results.exists():
    study_args += ["--reviews", review_results]
run_platform(*study_args)
print("Download selected reviewed artifacts and reports; do not commit caches, credentials, private data, or raw private outputs.")